# 12 · Benchmark de mercado: tarjeta de crédito en Colombia con datos públicos

**Pregunta de decisión.** ¿La cartera de la entidad está mejor o peor que el mercado, y contra quién conviene compararse?

Los notebooks 02–11 miran la cartera por dentro. El comité también pregunta por fuera: si el mercado subió la mora, si un competidor gana participación con riesgo más alto, si el precio está cerca de la usura o si la plaza donde crecemos es cara. Este notebook arma ese contexto **solo con datos públicos y agregados** de la Superintendencia Financiera (SFC) publicados en datos.gov.co:

1. inventario de lo que es público, lo que no y lo que solo existe en informes;
2. controles de calidad y conciliación entre fuentes;
3. tamaño, concentración y crecimiento;
4. uso: cupo, utilización, avances y persona jurídica;
5. riesgo: mora por altura, segmento de ingreso, castigos y un indicador que corrige el efecto del castigo;
6. precio frente a la usura, deudores nuevos y geografía;
7. cómo ubicar los números propios, incluida la conversión aproximada de una curva de cosecha (EVER) a mora de stock.

> **Fuentes:** datos abiertos de la SFC, información agregada por entidad. `mercado_publico.py` descarga y limpia; este notebook lee la copia guardada en `datos_publicos/` para no depender de la red. Para actualizar: `python mercado_publico.py`. Las cifras son las que publica la SFC y pueden tener retransmisiones posteriores.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
BASE = Path.cwd()
if not (BASE / 'decision_core.py').exists():
    root = next(p for p in [BASE, *BASE.parents] if (p / '03-financial-risk').is_dir())
    BASE = root / '03-financial-risk/01-credit-risk/credit-vintage-snapshot-lab'
sys.path.insert(0, str(BASE))
import mercado_publico as mp
plt.rcParams.update({'figure.figsize': (10, 4.6), 'font.size': 10.5, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .18,
                     'axes.axisbelow': True, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left'})
AZUL, OCRE, GRIS, ROJO = '#25658c', '#d4854a', '#9aa5b1', '#b5443b'
SIM = 'Datos simulados · no representa una cartera'
pd.options.display.float_format = '{:,.4f}'.format

def leer(nombre):
    """Lee una plantilla local; las del repositorio solo tienen encabezados."""
    return pd.read_csv(BASE / nombre, encoding='utf-8-sig')

def pct(x, d=1):
    return 'n/d' if pd.isna(x) else f'{100 * x:.{d}f} %'

def cop(x):
    return 'n/d' if pd.isna(x) else f'COP {x / 1e6:,.1f} M'

In [ ]:
D = mp.cargar(BASE / 'datos_publicos')
META = D['meta']
tar, car, tibc, des, geo = D['tarjetas'], mp.indicadores_mora(D['cartera']), D['tibc'], D['desembolsos'], D['geo']
FUENTE = 'Fuente: SFC, datos.gov.co · cálculos propios'
print(f"Descarga: {META['descargado']} | corte tarjetas {META['corte_tarjetas']} | cartera {META['corte_cartera']} | "
      f"desembolsos {META['corte_desembolsos']}")
print(f'Entidades con tarjeta de crédito: {tar.loc[tar.fecha == tar.fecha.max(), "entidad"].nunique()} | '
      f'meses: {tar.fecha.nunique()} ({tar.fecha.min():%Y-%m} a {tar.fecha.max():%Y-%m})')

## 1 · Qué hay público y qué no

El hallazgo más útil del inventario es una ausencia: **no existe una fuente pública de cosechas.** Ningún conjunto abierto publica EVER30@MOB3, curvas por cohorte, tasas de aprobación o desempeño por score. Lo público es **stock** (saldo y mora a cada corte) y **flujo** (desembolsos, compras, avances, castigos). El benchmark tiene que traducir entre las dos formas de mirar la cartera, y la sección 7 explica cómo.

In [ ]:
otras = pd.DataFrame([
    ('TransUnion · Colombia Credit Industry Insights (CIIR)', 'Trimestral, PDF', 'Mora 60+ por saldo y por consumidor, originaciones por nivel de riesgo y generación',
     'Única fuente pública con originación por riesgo; no descarga automatizada; definiciones propias del buró'),
    ('Experian/Datacrédito · informes de mercado', 'Irregular, PDF y prensa', 'Tendencias de originación y mora por producto', 'Útil como contraste; sin series'),
    ('Banco de la República · Reporte de Estabilidad Financiera', 'Semestral, PDF y anexos', 'Mora, castigos, cosechas de consumo agregadas, carga financiera de hogares',
     'Incluye análisis de cosechas por modalidad; ver anexos estadísticos'),
    ('Banca de las Oportunidades y SFC · Reporte de Inclusión Financiera', 'Anual y trimestral', 'Adultos con tarjeta por edad, sexo y municipio',
     'Penetración: cuántos adultos tienen tarjeta; base para dimensionar mercado potencial'),
    ('SFC · Inclusión financiera (kx2f-xjdq)', 'datos.gov.co', 'Productos y transacciones por municipio y entidad', 'Cubre conteos de crédito de consumo, no mora de tarjeta'),
    ('SFC · Captaciones y colocaciones por municipio (u2wk-tfe3)', 'datos.gov.co', 'Cartera por municipio y calificación', 'No separa tarjeta; sirve para el contexto territorial de consumo'),
    ('DANE · GEIH', 'Mensual', 'Desempleo nacional y por ciudad', 'Macro para el hazard del notebook 07'),
    ('Asobancaria · Banca y Economía', 'Semanal, PDF', 'Análisis sectoriales', 'Contexto cualitativo'),
], columns=['fuente', 'formato', 'contenido', 'uso en el laboratorio'])
display(Markdown('**Usadas aquí (descarga reproducible):**'))
display(mp.FUENTES[['id', 'nombre', 'granularidad', 'desde', 'contenido']])
display(Markdown('**Complementarias (lectura manual o sin serie abierta):**'))
display(otras)

## 2 · Controles antes de comparar

Una comparación entre entidades se cae por tres errores de datos: meses faltantes, nombres que cambian (fusiones y marcas nuevas) y dos fuentes que miden "saldo de tarjeta" de forma distinta. Se revisa cada uno:

- **Llave estable.** Se usa el código SFC (tipo × código), no el nombre. Scotiabank Colpatria aparece hoy como Davibank y Procredit como Ban100, con el mismo código.
- **Conciliación de saldo.** El formato de tarjetas (h2jg) incluye tarjetas empresariales; la distribución de cartera (rvii) es solo consumo. El cociente debe ser algo mayor que 1 y estable.
- **Cuadre interno.** En rvii, vigente + vencido debe igualar el saldo.

In [ ]:
tc = car[car.producto.str.contains('TARJETA') & (car.segmento == 'total')]
conc = (tar.groupby('fecha').saldo_tarjeta.sum() / tc.groupby('fecha').saldo.sum()).dropna()
meses = pd.date_range(tar.fecha.min(), tar.fecha.max(), freq='ME')
faltan = sorted(set(meses) - set(tar.fecha))
descuadre = (tc.descuadre > 0.001).mean()
ult = tar[tar.fecha == tar.fecha.max()]
solo_h2 = set(ult.entidad) - set(tc[tc.fecha == tc.fecha.max()].entidad)
display(pd.DataFrame({
    'control': ['Meses faltantes en tarjetas', 'Saldo h2jg / saldo rvii (mediana)', 'Rango del cociente',
                'Filas rvii con descuadre > 0,1 %', 'Entidades del último corte sin mora publicada'],
    'resultado': [len(faltan), f'{conc.median():.3f}', f'{conc.min():.3f} – {conc.max():.3f}', pct(descuadre, 2),
                  ', '.join(sorted(solo_h2)) or 'ninguna'],
}))
assert not faltan, 'Hay meses sin datos de tarjetas'
assert 1.0 < conc.median() < 1.15, 'Las dos fuentes de saldo no concilian dentro del rango esperado'
assert descuadre < 0.02

In [ ]:
itau = ult.merge(tc[tc.fecha == tc.fecha.max()][['entidad', 'saldo']], on='entidad', how='left')
itau['consumo_sobre_total'] = itau.saldo / itau.saldo_tarjeta
raras = itau[itau.consumo_sobre_total < 0.8].sort_values('saldo_tarjeta', ascending=False)
lectura = (f'El saldo del formato de tarjetas supera al de consumo en {pct(conc.median() - 1)} de mediana, '
           'diferencia coherente con las tarjetas empresariales que solo aparecen en h2jg. ')
if len(raras):
    lectura += ('Las entidades con cartera sobre todo empresarial (' + ', '.join(raras.entidad.head(4)) +
                ') no deben compararse en mora de consumo: su saldo de consumo es una fracción pequeña de su tarjeta.')
display(Markdown('**Lectura.** ' + lectura))

## 3 · Tamaño, concentración y crecimiento

In [ ]:
mk = mp.indicadores_tarjeta(tar.groupby('fecha', as_index=False).sum(numeric_only=True))
f1 = mk.fecha.max(); f0 = f1 - pd.DateOffset(years=1); f0 = mk.fecha[mk.fecha <= f0].max()
a, b = mk.set_index('fecha').loc[f1], mk.set_index('fecha').loc[f0]
resumen = pd.DataFrame({
    'indicador': ['Tarjetas vigentes (M)', 'Saldo (COP billones)', 'Cupo total (COP billones)', 'Compras 12 m (COP billones)',
                  'Avances 12 m (COP billones)', 'Castigos de capital 12 m (COP billones)'],
    f'{f1:%Y-%m}': [a.tarjetas_vigentes / 1e6, a.saldo_tarjeta / 1e12, (a.saldo_tarjeta + a.cupo_no_utilizado) / 1e12,
                   mk.set_index('fecha').monto_compras.loc[:f1].tail(12).sum() / 1e12,
                   mk.set_index('fecha').monto_avances.loc[:f1].tail(12).sum() / 1e12,
                   mk.set_index('fecha').castigos_capital.loc[:f1].tail(12).sum() / 1e12],
})
resumen['variación anual'] = [pct(a.tarjetas_vigentes / b.tarjetas_vigentes - 1), pct(a.saldo_tarjeta / b.saldo_tarjeta - 1),
                              pct((a.saldo_tarjeta + a.cupo_no_utilizado) / (b.saldo_tarjeta + b.cupo_no_utilizado) - 1), '', '', '']
display(resumen.style.format({f'{f1:%Y-%m}': '{:,.2f}'}).hide(axis='index'))

In [ ]:
cuota = ult.groupby('entidad').saldo_tarjeta.sum().sort_values(ascending=False)
cuota = cuota / cuota.sum()
HHI = mp.hhi(cuota)
ult_a = tar[tar.fecha == f0].groupby('entidad').saldo_tarjeta.sum(); ult_a = ult_a / ult_a.sum()
tabla = pd.DataFrame({'participación saldo': cuota, 'hace 12 m': ult_a.reindex(cuota.index)})
tabla['cambio (pp)'] = 100 * (tabla['participación saldo'] - tabla['hace 12 m'])
tarj = ult.groupby('entidad').tarjetas_vigentes.sum()
tabla['saldo por tarjeta (COP M)'] = ult.groupby('entidad').saldo_tarjeta.sum() / tarj / 1e6
display(tabla.head(12).style.format({'participación saldo': '{:.1%}', 'hace 12 m': '{:.1%}', 'cambio (pp)': '{:+.2f}',
                                    'saldo por tarjeta (COP M)': '{:,.2f}'}))
top = tabla['cambio (pp)'].idxmax(); peor = tabla['cambio (pp)'].idxmin()
display(Markdown(
    f'**Lectura.** Las cinco primeras concentran {pct(cuota.head(5).sum())} del saldo; el HHI es {HHI:,.0f} '
    f'({"moderado" if 1500 <= HHI < 2500 else "bajo" if HHI < 1500 else "alto"} en la escala de 10.000). '
    f'En el último año ganó más participación **{top}** ({tabla.loc[top, "cambio (pp)"]:+.2f} pp) y perdió más '
    f'**{peor}** ({tabla.loc[peor, "cambio (pp)"]:+.2f} pp). Un competidor que gana cuota rápido es el primer '
    'candidato a revisar en mora y castigo: crecer abre cosechas jóvenes que todavía no muestran su riesgo.'))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
s = tar.groupby('fecha').agg(tarj=('tarjetas_vigentes', 'sum'), saldo=('saldo_tarjeta', 'sum'))
ax[0].plot(s.index, s.tarj / 1e6, color=AZUL); ax[0].set_title('Tarjetas vigentes (millones)')
ax2 = ax[0].twinx(); ax2.plot(s.index, s.saldo / 1e12, color=OCRE); ax2.set_ylabel('saldo COP billones', color=OCRE); ax2.grid(False)
tipo = tar.assign(tipo=tar.tipo_entidad.map(mp.TIPOS_ENTIDAD)).pivot_table(index='fecha', columns='tipo', values='saldo_tarjeta', aggfunc='sum')
(tipo.div(tipo.sum(axis=1), axis=0) * 100).plot(ax=ax[1], color=[AZUL, OCRE, GRIS])
ax[1].set_title('Participación del saldo por tipo de entidad (%)'); ax[1].set_xlabel('')
fig.text(0.01, -0.02, FUENTE, fontsize=8, color='gray'); plt.tight_layout(); plt.show()
cf = tipo.div(tipo.sum(axis=1), axis=0)['Compañía de financiamiento']
display(Markdown(f'**Lectura.** Las compañías de financiamiento pasaron de {pct(cf.iloc[0])} a {pct(cf.iloc[-1])} del saldo. '
                 'Ahí están emisores digitales y de retail que originan fuera de la red bancaria; '
                 'son el benchmark relevante para una estrategia de originación digital como TPV.'))

## 4 · Uso: cupo, utilización y avances

Utilización $=\dfrac{\text{saldo}}{\text{saldo}+\text{cupo no utilizado}}$. Peso de avances $=\dfrac{\text{monto avances}}{\text{monto avances}+\text{monto compras}}$ en el mes. Los dos se relacionan con la mora: la utilización alta reduce el colchón del cliente y el avance es el uso que el notebook 10 estudia como señal o causa.

In [ ]:
ind = mp.indicadores_tarjeta(tar)
grandes = cuota.index[:12]
u = ind[(ind.fecha == f1) & ind.entidad.isin(grandes)].set_index('entidad').reindex(grandes)
uso = u[['utilizacion', 'cupo_promedio', 'peso_avances_monto', 'ticket_compra', 'participacion_juridica']].copy()
uso.loc['Mercado'] = mk.set_index('fecha').loc[f1, uso.columns]
display(uso.style.format({'utilizacion': '{:.1%}', 'cupo_promedio': lambda x: f'{x/1e6:,.1f} M', 'peso_avances_monto': '{:.1%}',
                          'ticket_compra': lambda x: f'{x/1e3:,.0f} mil', 'participacion_juridica': '{:.1%}'}))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
m2 = mk.set_index('fecha')
ax[0].plot(m2.index, m2.utilizacion * 100, color=AZUL); ax[0].set_title('Utilización del cupo, mercado (%)')
ax[1].plot(m2.index, m2.peso_avances_monto.rolling(3).mean() * 100, color=OCRE); ax[1].set_title('Peso de avances en el monto transado, media móvil 3 m (%)')
fig.text(0.01, -0.02, FUENTE, fontsize=8, color='gray'); plt.tight_layout(); plt.show()
hi = uso.drop('Mercado').peso_avances_monto.idxmax()
display(Markdown(f'**Lectura.** La utilización del mercado está en {pct(m2.utilizacion.iloc[-1])} y el avance pesa '
                 f'{pct(m2.peso_avances_monto.iloc[-1])} del monto transado. Entre los grandes, **{hi}** tiene el mayor peso de avances '
                 f'({pct(uso.loc[hi, "peso_avances_monto"])}). Una política de avance (notebook 10) se compara contra este rango, '
                 'no contra cero.'))

## 5 · Riesgo: mora por altura, ingreso y castigos

La SFC considera vencida la cartera de consumo con **más de 30 días**, el mismo umbral del EVER30 del laboratorio. Se calculan:

- $ICV_{30}=\dfrac{\text{saldo con mora}>30}{\text{saldo}}$, y análogos a 60 y 90 días;
- tasa de castigo anual $=\dfrac{\text{castigos de capital 12 m}}{\text{saldo promedio 12 m}}$;
- **mora ajustada por castigo** $=\dfrac{\text{mora}>30+\text{castigos 12 m}}{\text{saldo}+\text{castigos 12 m}}$.

El ajuste importa porque el ICV es un cociente de stock: una entidad que castiga rápido saca la mora del numerador y del denominador y se ve mejor de lo que es. La comparación honesta entre entidades usa las dos medidas.

In [ ]:
tcm = tc.groupby('fecha', as_index=False)[['saldo', 'mora30', 'mora60', 'mora90']].sum()
for k in ('30', '60', '90'):
    tcm[f'icv{k}'] = tcm[f'mora{k}'] / tcm.saldo
otros = car[(car.segmento == 'total') & ~car.producto.str.contains('TARJETA')].groupby(['fecha', 'producto'])[['mora30', 'saldo']].sum()
otros = (otros.mora30 / otros.saldo).unstack()
seg = car[car.producto.str.contains('TARJETA') & (car.segmento != 'total')].groupby(['fecha', 'segmento'])[['mora30', 'saldo']].sum()
seg = (seg.mora30 / seg.saldo).unstack()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(tcm.fecha, tcm.icv30 * 100, color=ROJO, lw=2.2, label='Tarjeta >30')
ax[0].plot(tcm.fecha, tcm.icv60 * 100, color=ROJO, lw=1.2, ls='--', label='Tarjeta >60')
for col, c in zip(['LIBRE INVERSION', 'LIBRANZA', 'VEHICULO'], [AZUL, GRIS, OCRE]):
    ax[0].plot(otros.index, otros[col] * 100, color=c, lw=1, label=col.title() + ' >30')
ax[0].set_title('Mora por saldo, consumo (%)'); ax[0].legend(fontsize=8, ncol=2)
for col, c in zip(seg.columns, [ROJO, AZUL]):
    ax[1].plot(seg.index, seg[col] * 100, color=c, label=col.replace('_', ' '))
ax[1].set_title('Tarjeta: mora >30 por ingreso del titular (%)'); ax[1].legend(fontsize=8)
fig.text(0.01, -0.02, FUENTE, fontsize=8, color='gray'); plt.tight_layout(); plt.show()
pico = tcm.loc[tcm.icv30.idxmax()]
rec = tcm[tcm.fecha >= '2022-01-01']; pico2 = rec.loc[rec.icv30.idxmax()]
display(Markdown(
    f'**Lectura.** La mora >30 de tarjeta tuvo su máximo en {pico.fecha:%Y-%m} ({pct(pico.icv30)}, pandemia) y un segundo pico en '
    f'{pico2.fecha:%Y-%m} ({pct(pico2.icv30)}); hoy está en {pct(tcm.icv30.iloc[-1])}. '
    f'El segmento de ingreso hasta 2 SMMLV corre {(seg.iloc[-1].ingreso_hasta_2_smmlv - seg.iloc[-1].ingreso_superior_2_smmlv) * 100:+.1f} pp '
    f'frente al superior. Las cosechas originadas antes del pico de {pico2.fecha:%Y} maduraron en el peor momento del ciclo reciente: '
    'compararlas con las posteriores sin el componente macro del notebook 07 atribuye a la originación lo que fue del entorno.'))

In [ ]:
cast = tar[['tipo_entidad', 'codigo_entidad', 'entidad', 'fecha', 'castigos_capital', 'saldo_tarjeta']].copy()
cast['castigos_capital'] = cast.castigos_capital.fillna(0)
cast['tasa_castigo'] = mp.anualizar_flujo(cast, 'castigos_capital', 'saldo_tarjeta', ['tipo_entidad', 'codigo_entidad'])
cast['castigo_12m'] = cast.groupby(['tipo_entidad', 'codigo_entidad']).castigos_capital.transform(lambda s: s.rolling(12, min_periods=12).sum())
r = tc[['tipo_entidad', 'codigo_entidad', 'fecha', 'saldo', 'mora30', 'icv30', 'icv60']].merge(
    cast[['tipo_entidad', 'codigo_entidad', 'fecha', 'entidad', 'tasa_castigo', 'castigo_12m', 'saldo_tarjeta']],
    on=['tipo_entidad', 'codigo_entidad', 'fecha'], how='inner')
r['mora_ajustada'] = (r.mora30 + r.castigo_12m) / (r.saldo + r.castigo_12m)
fr = r.fecha.max()
rk = r[(r.fecha == fr) & (r.saldo > 1e11) & (r.saldo / r.saldo_tarjeta > 0.8)].dropna(subset=['tasa_castigo']).set_index('entidad')
rk['puesto ICV30'] = rk.icv30.rank().astype(int)
rk['puesto ajustado'] = rk.mora_ajustada.rank().astype(int)
rk['salto'] = rk['puesto ajustado'] - rk['puesto ICV30']
display(rk.sort_values('icv30')[['saldo', 'icv30', 'icv60', 'tasa_castigo', 'mora_ajustada', 'puesto ICV30', 'puesto ajustado', 'salto']]
        .style.format({'saldo': lambda x: f'{x/1e12:,.2f} B', 'icv30': '{:.1%}', 'icv60': '{:.1%}', 'tasa_castigo': '{:.1%}', 'mora_ajustada': '{:.1%}'}))
mov = rk.salto.abs().idxmax()
sentido = 'mejora' if rk.loc[mov, 'salto'] < 0 else 'empeora'
med_cast = np.average(rk.tasa_castigo, weights=rk.saldo)
display(Markdown(
    f'**Lectura.** Con corte {fr:%Y-%m}, entre emisores con más de COP 0,1 billones de saldo de consumo. '
    f'El castigo cambia el orden: **{mov}** {sentido} del puesto {rk.loc[mov, "puesto ICV30"]} al {rk.loc[mov, "puesto ajustado"]} '
    f'cuando se suma lo castigado en 12 meses (castiga {pct(rk.loc[mov, "tasa_castigo"])} anual frente a {pct(med_cast)} del grupo). '
    'Quien castiga poco y tiene ICV alto sube en la lista; quien castiga mucho y luce bien en ICV baja. '
    'Para el comité, el benchmark de riesgo se presenta con las dos columnas; comparar solo el ICV premia castigar rápido.'))
assert (rk.mora_ajustada >= rk.icv30 - 1e-12).all()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(rk.icv30 * 100, rk.tasa_castigo * 100, s=rk.saldo / rk.saldo.max() * 900, color=AZUL, alpha=.45, edgecolor='white')
etiquetas = set(rk.saldo.nlargest(3).index) | set(rk[(rk.tasa_castigo > .10) | (rk.icv30 > .06)].index)
for j, (n, f) in enumerate(rk.loc[sorted(etiquetas, key=lambda e: rk.loc[e, 'icv30'])].iterrows()):
    ax.annotate(n.split(' (')[0], (f.icv30 * 100, f.tasa_castigo * 100), fontsize=8,
                xytext=(6, 4 if j % 2 == 0 else -11), textcoords='offset points')
mx = tcm.set_index('fecha').loc[fr]
ax.axvline(mx.icv30 * 100, color=GRIS, ls=':', lw=1)
ax.set_xlabel('Mora >30 por saldo (%)'); ax.set_ylabel('Castigo anual de capital / saldo promedio (%)')
ax.set_title(f'Mora y castigo por emisor, {fr:%Y-%m} (tamaño = saldo; se rotulan los 3 mayores y los atípicos)')
fig.text(0.01, -0.02, FUENTE, fontsize=8, color='gray'); plt.tight_layout(); plt.show()

### Contraste con el buró

TransUnion publica la mora 60+ por saldo de tarjeta en su informe trimestral (CIIR). Su universo es el de los reportes al buró y su definición de saldo no es la de la SFC, así que el contraste sirve para validar **dirección y orden de magnitud**, no para cuadrar. Los valores externos se copian a mano del informe y quedan con su fuente.

In [ ]:
buro = pd.DataFrame([
    ('2025-12-31', 0.0434, 'TransUnion CIIR 4T-2025 (mora 60+ por saldo, tarjeta)'),
    ('2026-03-31', 0.0412, 'TransUnion CIIR 1T-2026 (mora 60+ por saldo, tarjeta)'),
], columns=['fecha', 'mora60_buro', 'fuente'])
buro['fecha'] = pd.to_datetime(buro.fecha)
cmp_ = buro.merge(tcm[['fecha', 'icv60']], on='fecha', how='left')
cmp_['diferencia (pp)'] = 100 * (cmp_.icv60 - cmp_.mora60_buro)
display(cmp_.style.format({'mora60_buro': '{:.2%}', 'icv60': '{:.2%}', 'diferencia (pp)': '{:+.2f}'}).hide(axis='index'))
mismo_sentido = np.sign(cmp_.icv60.diff().iloc[-1]) == np.sign(cmp_.mora60_buro.diff().iloc[-1])
display(Markdown(f'**Lectura.** La SFC y el buró {"se mueven en el mismo sentido" if mismo_sentido else "se mueven en sentidos opuestos"} '
                 f'entre los dos trimestres, con una brecha de nivel de {cmp_["diferencia (pp)"].abs().mean():.1f} pp. '
                 'Una brecha estable es esperable por universo y definición; una que cambia de signo pide revisar la fuente antes de usarla.'))

## 6 · Precio, deudores nuevos y geografía

La SFC publica cada semana el monto desembolsado y la tasa promedio por producto y **clase de deudor** (nuevo o existente en la entidad). En tarjeta, "desembolso" es la utilización del cupo en la semana, así que mide actividad y precio, no originación de cuentas. La tasa se pondera por monto.

In [ ]:
d = des.copy()
d['mes'] = d.semana.dt.to_period('M').dt.to_timestamp('M')
pm = d[d['producto'] != 'empresarial'].groupby(['mes', 'producto']).agg(monto=('monto', 'sum'), tx=('tasa_x_monto', 'sum'))
pm['tasa'] = pm.tx / pm.monto / 100
pm = pm.tasa.unstack()
usura = mp.usura_mensual(tibc)
nuevo = d.groupby(['mes', 'deudor_nuevo']).monto.sum().unstack().fillna(0)
nuevo_share = nuevo[True] / nuevo.sum(axis=1)
inicio_campo = nuevo_share[nuevo_share > 0].index.min()   # la clase de deudor solo se reporta desde esta fecha
nuevo_share = nuevo_share[nuevo_share.index >= inicio_campo]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
for col, c in zip(pm.columns, [ROJO, AZUL]):
    ax[0].plot(pm.index, pm[col] * 100, color=c, label=col.replace('_', ' '))
uu = usura.reindex(pm.index)
ax[0].plot(uu.index, uu * 100, color='black', lw=1, ls='--', label='usura consumo')
ax[0].set_title('Tasa efectiva ponderada de tarjeta (% E.A.)'); ax[0].legend(fontsize=8)
ax[1].plot(nuevo_share.index, nuevo_share * 100, color=OCRE); ax[1].set_title(f'Monto de deudores nuevos en la entidad (% del total, desde {inicio_campo:%Y-%m})')
fig.text(0.01, -0.02, FUENTE, fontsize=8, color='gray'); plt.tight_layout(); plt.show()
brecha = (uu - pm.max(axis=1)).dropna()
estado = 'pegada al techo' if brecha.iloc[-1] < 0.03 else 'cerca del techo'
display(Markdown(
    f'**Lectura.** La tasa de tarjeta está {estado}: en el último mes la brecha entre la usura y el producto más caro es de '
    f'{brecha.iloc[-1] * 100:.1f} pp (mínimo del periodo {brecha.min() * 100:.1f} pp, en {brecha.idxmin():%Y-%m}). Es un promedio ponderado: '
    'los segmentos de mayor riesgo se cobran más cerca de la usura, así que para ellos el riesgo no se compensa con tasa. '
    'Por eso la economía del notebook 08 se ajusta por **corte, cupo y comisiones**, no solo por precio. '
    f'Los deudores nuevos pesan {pct(nuevo_share.iloc[-1])} del monto del último mes.'))

In [ ]:
g = geo[(geo['producto'] != 'empresarial') & (geo.departamento != 'Sin municipio')].groupby('departamento').agg(
    monto=('monto', 'sum'), tx=('tasa_x_monto', 'sum'), n=('n', 'sum'))
gn = geo[(geo['producto'] != 'empresarial') & geo.deudor_nuevo].groupby('departamento').monto.sum()
g['tasa'] = g.tx / g.monto / 100
g['participación'] = g.monto / g.monto.sum()
g['peso deudor nuevo'] = (gn / g.monto).reindex(g.index)
g = g.sort_values('monto', ascending=False)
display(Markdown(f"Trimestre {geo.trimestre.iloc[0]}"))
display(g[['participación', 'tasa', 'peso deudor nuevo']].head(15).style.format('{:.1%}'))
display(Markdown(f'**Lectura.** Bogotá, Antioquia y Valle concentran {pct(g["participación"].head(3).sum())} del monto. '
                 'Para el análisis de ubicación del laboratorio, la mezcla geográfica de la entidad se compara con esta distribución: '
                 'crecer en un departamento donde el mercado no está es una apuesta, no una consecuencia.'))

## 7 · Ubicar a la entidad

### 7.1 Con datos de la entidad

`plantilla_benchmark_entidad.csv` recibe los indicadores propios en las mismas definiciones: mes, saldo, mora >30 y >60 por saldo, castigos de capital del mes, utilización y peso de avances. Con eso, cada indicador se ubica como percentil ponderado por saldo dentro del mercado del mismo mes.

In [ ]:
propia = leer('plantilla_benchmark_entidad.csv')
print(f'Filas propias: {len(propia):,}')

def ubicar(propia_mes, mercado_mes):
    """Percentil ponderado por saldo de cada indicador propio en el mercado del mismo corte."""
    out = {}
    for k in ('icv30', 'icv60', 'mora_ajustada', 'tasa_castigo'):
        if k in propia_mes and pd.notna(propia_mes[k]):
            out[k] = mp.percentil_en_mercado(propia_mes[k], mercado_mes[k], mercado_mes['saldo'])
    return pd.Series(out, name='percentil (0 = mejor)')

if propia.empty:
    print('PENDIENTE: cargar los indicadores propios con las definiciones de la sección 5 '
          '(la plantilla solo tiene encabezados).')

### 7.2 Demostración: una entidad hipotética

Indicadores inventados para una entidad mediana, ubicados en el mercado real del último corte. Sirve para ver la salida, no describe a ningún emisor.

In [ ]:
merc = r[(r.fecha == fr) & (r.saldo / r.saldo_tarjeta > 0.8)].dropna(subset=['tasa_castigo'])
hipotetica = pd.Series({'icv30': 0.058, 'icv60': 0.041, 'tasa_castigo': 0.09})
hipotetica['mora_ajustada'] = (hipotetica.icv30 + hipotetica.tasa_castigo) / (1 + hipotetica.tasa_castigo)
pos = ubicar(hipotetica, merc)
display(pd.DataFrame({'valor hipotético': hipotetica, 'mediana ponderada mercado': [
    np.average(merc[k], weights=merc.saldo) for k in hipotetica.index], 'percentil': pos}).style.format('{:.1%}'))
assert pos.between(0, 1).all()

### 7.3 De la cosecha al stock

El laboratorio mide **entrada a mora por cosecha** (EVER); la SFC publica **stock de mora**. No son comparables directamente: el stock depende de la curva de entrada, de qué tan rápido sale la mora del balance (castigo o cura) y del crecimiento, porque una cartera que crece tiene más cosechas jóvenes.

`mp.mora_stock_estacionaria` aproxima el stock de estado estacionario que implica una curva de entrada. No es una conversión exacta; responde una pregunta concreta: **¿la curva de cosecha que estamos viendo es consistente con el ICV que publicamos?** Si no lo es, falta un componente (cura, reestructuración, castigo tardío) o hay un problema de datos.

**Hipótesis inicial, que resultó falsa en general:** "crecer siempre baja la mora de stock, porque las cosechas jóvenes tienen poca mora". La grilla muestra que depende del perfil de mora por edad. Si la mora sale lento, se acumula con la edad y crecer la diluye. Si entra temprano y sale rápido, las cosechas jóvenes son las que más mora tienen y crecer **sube** el stock.

In [ ]:
edades = np.arange(1, 25)
curva = 0.13 * (1 - np.exp(-edades / 7))   # curva ilustrativa: ~4,5 % de entrada a MOB 3 y ~13 % a largo plazo
salidas, crec = (0.0, 0.05, 0.10, 0.15, 0.20), (0.0, 0.01, 0.02)
grid = pd.DataFrame({g: {ro: mp.mora_stock_estacionaria(curva, ro, g) for ro in salidas} for g in crec})
grid.index.name = 'salida mensual de la mora'; grid.columns = [f'crecimiento {c:.0%}/mes' for c in grid.columns]
display(grid.style.format('{:.1%}'))
fig, ax = plt.subplots(figsize=(9, 3.6))
for ro, c in zip((0.0, 0.10, 0.20), (GRIS, AZUL, ROJO)):
    ax.plot(edades, 100 * mp.perfil_mora_por_edad(curva, ro), color=c, label=f'salida {ro:.0%}/mes')
ax.set_xlabel('edad de la cosecha (meses)'); ax.set_ylabel('% del saldo en mora >30'); ax.legend(fontsize=8)
ax.set_title('Perfil de mora por edad con la misma curva de entrada'); fig.text(0.01, -0.03, 'Curva ilustrativa', fontsize=8, color='gray')
plt.tight_layout(); plt.show()
efecto = grid.iloc[:, -1] - grid.iloc[:, 0]
diluye = efecto[efecto < 0].index.tolist(); infla = efecto[efecto > 0].index.tolist()
obj = tcm.icv30.iloc[-1]
display(Markdown(
    f'**Lectura.** Con esta curva ilustrativa (EVER a MOB 3 de {pct(curva[2])}), el stock implicado va de {pct(grid.values.min())} '
    f'a {pct(grid.values.max())}; el mercado publica {pct(obj)}. Crecer al 2 % mensual baja el stock con salidas de '
    f'{", ".join(f"{x:.0%}" for x in diluye) or "ninguna"} y lo sube con salidas de {", ".join(f"{x:.0%}" for x in infla) or "ninguna"}. '
    'Por eso no hay una corrección única por crecimiento: el benchmark de stock se lee junto con la curva de cosecha y la velocidad de castigo '
    'de cada emisor, y la comparación justa entre emisores que crecen distinto es la de curvas, no la de stock.'))
assert grid.iloc[:, 0].is_monotonic_decreasing
for ro in salidas:   # el signo del efecto del crecimiento sigue la pendiente del perfil por edad
    pend = np.polyfit(edades, mp.perfil_mora_por_edad(curva, ro), 1)[0]
    assert np.sign(efecto.loc[ro]) == -np.sign(pend) or abs(efecto.loc[ro]) < 5e-4

## 8 · Qué llevar al comité

| Pregunta | Indicador público | Cuidado |
|---|---|---|
| ¿Estamos peor que el mercado? | ICV30, ICV60 y mora ajustada por castigo, percentil ponderado | comparar solo emisores con cartera de consumo comparable |
| ¿El mercado empeoró? | serie de ICV y castigo del mercado; contraste con el buró | la mora de stock rezaga a la originación varios meses |
| ¿Quién crece y con qué riesgo? | cambio de participación y saldo por tarjeta | crecer cambia el ICV por mezcla de edades, en un sentido que depende de la curva; mirar castigo y esperar maduración |
| ¿Nuestro precio tiene margen? | tasa ponderada frente a usura | el techo limita el precio; ajustar por corte y cupo |
| ¿Dónde crecer? | monto, tasa y peso de deudores nuevos por departamento | la ubicación propia se compara con la mezcla del mercado |
| ¿Nuestro uso es normal? | utilización y peso de avances | los avances altos son tema del notebook 10 |

**Lo que el dato público no da:** cosechas, aprobación, desempeño por score ni por canal. Para eso, la referencia es interna (notebooks 02–11) o el buró bajo contrato.